# Figuring out the dice math on the board game Masters of the Universe

In [2]:
import numpy as np
import pandas as pd
from itertools import product

In [3]:
# instantiating dice classes to store dice probabilities
class Die:
    def __init__(self, faces_probs):
        self.faces_probs = faces_probs
    def probability(self, result):
        return self.faces_probs.get(result, 0)
    def faces(self):
        return self.faces_probs.keys()

**The game has three types of dice**
| Color of the dice | #                                             | %                                                 |
|-------------------|-----------------------------------------------|---------------------------------------------------|
| Yellow            | 3 blanks<br>2x 1 axe<br>1x 2 axes             | 50% blank<br>33.33% 1 axe<br>16.66% 2 axes        |
| Orange            | 2 blanks<br>2x 1 axe<br>2x 2 axes             | 33% blanks<br>33% 1 axe<br>33% 2 axes             |
| Red               | 1 blank<br>2x 1 axe<br>2x 2 axes<br>1x 3 axes | 16% blank<br>33% 1 axe<br>33% 2 axes<br>16% 3 axes |

In [5]:
# instantiating each die with probabilities 
yellow_die = Die({0: 0.5, 1: 0.33, 2: 0.16})
orange_die = Die({0: 0.33, 1: 0.33, 2: 0.33})
red_die = Die({0: 0.16, 1: 0.33, 2: 0.33, 3: 0.16})

# determining joint probability of dice
def joint_prob(outcome, dice):
    prob = 1.0
    for die, result in zip(dice, outcome):
        prob *= die.probability(result)
    return prob

# determining whether the joint probability is higher than target
def total_prob_greater_than(target, dice):
    all_faces = [list(die.faces()) for die in dice]
    total = 0.0
    for outcome in product(*all_faces):
        s = sum(r if r != 0 else 0 for r in outcome)
        if s > target:
            total += joint_prob(outcome, dice)
    return round(total*100, 1)

## Attack power of He-Man, Orko and Teela

- He-Man: 3 orange dice along with one red die
- Orko: 2 yellow dice plus 1 orange die
- Teela: 3 yellow, 1 orange plus one red die

In [7]:
he_man_dice = [orange_die] * 3 + [red_die] * 1
orko_dice = [yellow_die] * 2 + [orange_die] * 1
teela_dice = [yellow_die] * 3 + [orange_die] * 1 + [red_die] * 1

characters_attack_power = []

for i in range(1,11):
    he_man_prob_success = total_prob_greater_than(i, he_man_dice)
    orko_prob_success = total_prob_greater_than(i, orko_dice)
    teela_prob_success = total_prob_greater_than(i, teela_dice)
    if (he_man_prob_success >= 4) or (orko_prob_success >= 4) or (teela_prob_success >= 4):
        per_target = {}
        per_target['target'] = i
        per_target['he_man_prob_success'] = he_man_prob_success
        per_target['orko_prob_success'] = orko_prob_success
        per_target['teela_prob_success'] = teela_prob_success
        characters_attack_power.append(per_target)
    else:
        break

pd.DataFrame(characters_attack_power)

,target,he_man_prob_success,orko_prob_success,teela_prob_success
0,1,91.6,69.6,90.2
1,2,83.4,41.6,81.3
2,3,68.1,18.4,65.6
3,4,47.5,5.2,45.6
4,5,27.0,0.8,26.4
5,6,11.7,0.0,12.4
6,7,3.5,0.0,4.5
